In [1]:
import numpy as np
from scipy.stats import rankdata
from hyppo.independence import Dcorr, Hsic
import time

In [2]:
def pseudo_obs(X):
    n = X.shape[0]
    U = np.zeros_like(X, dtype=float)
    for j in range(X.shape[1]):
        U[:, j] = rankdata(X[:, j]) / (n + 1.0)
    return U

def cvm_stat_multivariate(X, Z):
    Ux = pseudo_obs(X)
    Uz = pseudo_obs(Z)
    W = np.hstack([Ux, Uz])
    n, d = W.shape

    diff = 1 - np.maximum(W[:, None, :], W[None, :, :])
    term1 = np.sum(np.prod(diff, axis=2)) / n**2
    term2 = np.mean(np.prod(0.5 * (1 - W**2), axis=1))
    term3 = (1/3)**d

    return n * (term1 - 2*term2 + term3)

def cvm_test(X, Z, B=300):
    n = X.shape[0]
    T_obs = cvm_stat_multivariate(X, Z)
    T_perm = np.zeros(B)
    for b in range(B):
        perm = np.random.permutation(n)
        T_perm[b] = cvm_stat_multivariate(X, Z[perm])
    p = np.mean(T_perm >= T_obs)
    return T_obs, p

In [3]:
def generate_independent(n, dx, dz):
    X = np.random.normal(size=(n, dx))
    Z = np.random.normal(size=(n, dz))
    return X, Z

In [4]:
def generate_linear(n, dx, dz, rho=0.5):
    X = np.random.normal(size=(n, dx))
    noise = np.random.normal(size=(n, dz))
    Z = rho * X[:, :dz] + np.sqrt(1-rho**2) * noise
    return X, Z

In [5]:
def generate_quadratic(n):
    X = np.random.uniform(-1, 1, size=(n,1))
    Z = X**2 + 0.2*np.random.normal(size=(n,1))
    return X, Z

In [6]:
def compare_methods(X, Z):
    results = {}

    start = time.time()
    stat_cvm, p_cvm = cvm_test(X, Z)
    results["CvM"] = (p_cvm, time.time()-start)

    start = time.time()
    stat_dcor, p_dcor = Dcorr().test(X, Z)
    results["dCor"] = (p_dcor, time.time()-start)

    start = time.time()
    stat_hsic, p_hsic = Hsic().test(X, Z)
    results["HSIC"] = (p_hsic, time.time()-start)

    return results

In [7]:
np.random.seed(0)
n = 200

print("=== Independent ===")
X, Z = generate_independent(n, 1, 1)
print(compare_methods(X, Z))

print("\n=== Linear dependence ===")
X, Z = generate_linear(n, 1, 1)
print(compare_methods(X, Z))

print("\n=== Quadratic dependence ===")
X, Z = generate_quadratic(n)
print(compare_methods(X, Z))

=== Independent ===
{'CvM': (np.float64(0.2966666666666667), 0.27613210678100586), 'dCor': (np.float64(0.35121860552685147), 10.56725001335144), 'HSIC': (np.float64(0.5904285485547301), 0.0063860416412353516)}

=== Linear dependence ===
{'CvM': (np.float64(0.0), 0.28279590606689453), 'dCor': (np.float64(1.0318016625544157e-10), 0.0012631416320800781), 'HSIC': (np.float64(1.9416726084773016e-08), 0.0027878284454345703)}

=== Quadratic dependence ===
{'CvM': (np.float64(0.0), 0.28043389320373535), 'dCor': (np.float64(4.998211010858557e-07), 0.0011968612670898438), 'HSIC': (np.float64(2.8304551626774864e-10), 0.002946138381958008)}
